# SpecProbe — live fire on Colab

Gut `vllm.v1.sample.rejection_sampler.RejectionSampler.forward`, dump JSONL, prove the patch is **lossless** under greedy sampling, then either:

1. **Offline (recommended):** download `live.jsonl` → drop into your local Vite UI  
2. **Live stream:** start the FastAPI bridge on `:8787` and open it via Colab's native port forward

**Runtime → Change runtime type → GPU** (T4 minimum; A100 preferred for Llama-3-8B + EAGLE).

Constraints baked in: `pipeline_parallel_size=1`, `draft_tensor_parallel_size=1`.

## 0. Config

In [ ]:
# --- edit these ---
REPO_URL = "https://github.com/brian-mwirigi/spec-probe.git"  # or upload a zip (see cell 1b)
HF_TOKEN = ""  # or leave blank and login interactively / Colab secret HF_TOKEN

# Default: ungated models (works immediately on Colab T4)
TARGET_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
EAGLE_MODEL = None  # n-gram speculative only (no gated draft)

# Optional: Llama-3-8B + EAGLE (requires HF license acceptance — often takes hours/days)
# 1) Visit https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct → Agree and access
# 2) Visit the EAGLE draft page and accept if gated
# 3) Wait until your account shows Access granted, then uncomment:
# TARGET_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"
# EAGLE_MODEL = "yuhuili/EAGLE-LLaMA3-Instruct-8B"

PROMPT = """def fib(n):
    if n < 2:
        return n
    return fib(n-1) +"""

MAX_TOKENS = 48
JSONL_PATH = "/content/specprobe_traces/live.jsonl"
BRIDGE_PORT = 8787


## 1. Fix Colab's asyncio clash + install deps

Colab already owns an event loop. FastAPI / uvicorn / some vLLM paths will explode without `nest_asyncio`.

In [ ]:
import sys
import subprocess
import os

def pip_install(*packages: str) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

pip_install("nest_asyncio", "fastapi", "uvicorn[standard]", "pydantic", "huggingface_hub")

import nest_asyncio
nest_asyncio.apply()
print("nest_asyncio applied — Colab event loop is safe")

import torch
assert torch.cuda.is_available(), "Enable a GPU runtime (Runtime → Change runtime type → GPU)"
print("GPU:", torch.cuda.get_device_name(0), "| VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print("torch CUDA:", torch.version.cuda)


def install_vllm_for_colab() -> None:
    """PyPI vLLM is CUDA-13 linked; Colab is CUDA-12 → libcudart.so.13 ImportError.

    Install an official +cu129 wheel from GitHub Releases instead.
    """
    import platform
    import urllib.request

    # Uninstall any broken CUDA-13 wheel first
    subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "vllm"])

    arch = platform.machine()  # x86_64
    # abi3 wheel works across CPython 3.9+ including Colab's 3.12
    ver = "0.21.0"
    wheel = (
        f"https://github.com/vllm-project/vllm/releases/download/v{ver}/"
        f"vllm-{ver}+cu129-cp38-abi3-manylinux_2_34_{arch}.whl"
    )
    print("Installing Colab-compatible vLLM wheel:\n ", wheel)
    try:
        urllib.request.urlopen(wheel, timeout=20)
    except Exception as exc:
        raise RuntimeError(
            "Could not reach the cu129 vLLM wheel URL. "
            "Check network or pick another release from "
            "https://github.com/vllm-project/vllm/releases"
        ) from exc

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            wheel,
            "--extra-index-url",
            "https://download.pytorch.org/whl/cu129",
        ]
    )

    # Sanity import
    import vllm
    from vllm import LLM, SamplingParams  # noqa: F401
    print("vLLM OK:", vllm.__version__)


print("Defined install_vllm_for_colab() — run it in the next install cell")


## 1b. Install SpecProbe hook + vLLM

Prefer cloning your repo. If `REPO_URL` is still a placeholder, upload `SpecProbe` as a zip to `/content` and set `USE_UPLOAD = True`.

In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

# Match the GitHub repo folder name
root = Path("/content/spec-probe")
USE_UPLOAD = False  # True if you uploaded a zip to /content

# Always ensure build tooling exists (editable installs fail without it on Colab)
pip_install("setuptools", "wheel", "build")

if USE_UPLOAD:
    zips = list(Path("/content").glob("*.zip"))
    assert root.exists() or zips, "Upload the spec-probe zip/folder to /content first"
    if not root.exists() and zips:
        import zipfile
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall("/content")
        # handle either extracted as spec-probe/ or SpecProbe/
        if not root.exists():
            for cand in (Path("/content/SpecProbe"), Path("/content/spec-probe-main")):
                if cand.exists():
                    cand.rename(root)
                    break
else:
    # Re-clone if previous attempt left a broken tree
    pip_dir_check = root / "python" / "specprobe_hook"
    if root.exists() and not pip_dir_check.exists():
        print("Removing broken clone:", root)
        shutil.rmtree(root)
    if not root.exists():
        print("Cloning", REPO_URL, "→", root)
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(root)])

pip_dir = root / "python"
pkg = pip_dir / "specprobe_hook"
print("repo root:", root, "exists=", root.exists())
print("package:", pkg, "exists=", pkg.exists())
assert pkg.exists(), f"Missing {pkg}. Re-run after fixing REPO_URL / upload."

# Non-quiet install so failures are visible
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-e", f"{pip_dir}[bridge]"]
)

# Belt-and-suspenders: put the package on sys.path even if site-packages lags
if str(pip_dir) not in sys.path:
    sys.path.insert(0, str(pip_dir))

# vLLM: MUST use cu129 wheel on Colab (not plain pip install vllm)
install_vllm_for_colab()

from specprobe_hook import install, uninstall
print("spec-probe + vLLM ready")
print("install()", install)


## 2. Hugging Face auth

Needed if you pull **gated** weights (Llama). For the default Qwen path, login is optional.

If you see `GatedRepoError` / `403` on Llama:
1. Open https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct
2. Click **Agree and access repository** (same HF account as your token)
3. Wait until the page says you have access (not instant)
4. Re-run login + the losslessness cell


In [ ]:
from huggingface_hub import login

if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    try:
        from google.colab import userdata
        login(token=userdata.get("HF_TOKEN"))
    except Exception:
        login()  # interactive prompt
print("HF auth ok")

## 3. Helpers — build engine + greedy generate

> **Colab CUDA note:** plain `pip install vllm` pulls a CUDA-13 wheel and crashes with
> `libcudart.so.13`. The install cell uses the official `+cu129` release wheel instead.

Speculative config rules:
- **n-gram:** `{"method": "ngram", "prompt_lookup_max": N, "draft_tensor_parallel_size": 1}`
- **EAGLE:** `{"model": "...", "draft_tensor_parallel_size": 1}` — **no** `method` key
- Always `pipeline_parallel_size=1`


In [ ]:
import os
import gc
import json
from typing import Any, Optional

from vllm import LLM, SamplingParams


def speculative_config(eagle_model: Optional[str]) -> dict[str, Any]:
    if eagle_model:
        return {
            "model": eagle_model,
            "num_speculative_tokens": 5,
            "draft_tensor_parallel_size": 1,
        }
    return {
        "method": "ngram",
        "num_speculative_tokens": 5,
        "prompt_lookup_max": 4,
        "draft_tensor_parallel_size": 1,
    }


def build_llm(*, enable_spec: bool, eagle_model: Optional[str]) -> LLM:
    kwargs: dict[str, Any] = dict(
        model=TARGET_MODEL,
        trust_remote_code=True,
        dtype="bfloat16",
        max_model_len=2048,
        gpu_memory_utilization=0.90,
        tensor_parallel_size=1,
        pipeline_parallel_size=1,  # speculative decode + PP = crash
        enforce_eager=True,  # friendlier for Colab / patching
    )
    if enable_spec:
        kwargs["speculative_config"] = speculative_config(eagle_model)
    print("LLM kwargs:", {k: v for k, v in kwargs.items() if k != "speculative_config"})
    if enable_spec:
        print("speculative_config:", json.dumps(kwargs["speculative_config"]))
    return LLM(**kwargs)


def greedy_generate(llm: LLM, prompt: str, max_tokens: int = MAX_TOKENS) -> str:
    params = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=max_tokens)
    out = llm.generate([prompt], params)[0]
    return out.outputs[0].text


def nuke_llm(llm: Optional[LLM]) -> None:
    if llm is None:
        return
    del llm
    gc.collect()
    torch.cuda.empty_cache()


print("helpers ready")

## 4. Losslessness check (the trust builder)

Prove mathematically:

```text
greedy_with_specprobe  ==  greedy_without_specprobe
```

Same speculative config, temperature `0`. If the async hook corrupted the sampler, this fails.

In [ ]:
os.environ["SPECPROBE_DISABLE"] = "1"  # belt + suspenders while building baseline
uninstall()  # no-op if never installed

print("\n=== A) greedy WITHOUT SpecProbe patch ===")
llm_a = build_llm(enable_spec=True, eagle_model=EAGLE_MODEL)
text_a = greedy_generate(llm_a, PROMPT)
print(repr(text_a))
nuke_llm(llm_a)

print("\n=== B) greedy WITH SpecProbe patch ===")
os.environ.pop("SPECPROBE_DISABLE", None)
os.environ["SPECPROBE_JSONL"] = "/tmp/specprobe_lossless.jsonl"
os.environ["SPECPROBE_STRATEGY"] = "eagle" if EAGLE_MODEL else "ngram"
os.environ["SPECPROBE_PROMPT"] = PROMPT[:2000]
os.environ["SPECPROBE_RUN_ID"] = "colab_lossless"
Path(os.environ["SPECPROBE_JSONL"]).write_text("", encoding="utf-8")

assert install(), "SpecProbe patch failed to install — is vLLM v1 available?"
llm_b = build_llm(enable_spec=True, eagle_model=EAGLE_MODEL)
text_b = greedy_generate(llm_b, PROMPT)
print(repr(text_b))
nuke_llm(llm_b)

ok = text_a == text_b
print("\n" + ("PASS" if ok else "FAIL"), "greedy_with_specprobe == greedy_without_specprobe")
assert ok, (
    "Patch changed greedy output — do NOT ship. "
    "Check that schedule_extract never mutates draft_probs/target_logits in-place."
)
print("Losslessness OK — async extract did not break the sampler.")

## 5. Live fire — emit `live.jsonl`

This is the **offline fallback**: after this cell, download the file and drop it into your local SpecProbe UI (`public/traces/` or the file picker). No tunnel required.

In [ ]:
from pathlib import Path

out_dir = Path(JSONL_PATH).parent
out_dir.mkdir(parents=True, exist_ok=True)
Path(JSONL_PATH).write_text("", encoding="utf-8")

os.environ["SPECPROBE_JSONL"] = JSONL_PATH
os.environ["SPECPROBE_STRATEGY"] = "eagle" if EAGLE_MODEL else "ngram"
os.environ["SPECPROBE_PROMPT"] = PROMPT[:2000]
os.environ["SPECPROBE_RUN_ID"] = "colab_live"
os.environ.pop("SPECPROBE_DISABLE", None)

assert install(force=True)

llm = build_llm(enable_spec=True, eagle_model=EAGLE_MODEL)
# temperature>0 still fine for traces; use 0.8 to match the UI demo vibe
params = SamplingParams(temperature=0.8, top_p=0.95, max_tokens=MAX_TOKENS)
result = llm.generate([PROMPT], params)[0]
print("completion:\n", result.outputs[0].text)
nuke_llm(llm)

# give async extract worker a beat to flush D2H + JSONL
import time
time.sleep(1.5)

size = Path(JSONL_PATH).stat().st_size
n_lines = sum(1 for line in Path(JSONL_PATH).read_text(encoding="utf-8").splitlines() if line.strip())
print(f"\nwrote {JSONL_PATH} ({size} bytes, {n_lines} events)")
assert n_lines > 0, "No JSONL events — RejectionSampler.forward may not have been hit"

print("\nFirst event keys:")
first = json.loads(Path(JSONL_PATH).read_text(encoding="utf-8").splitlines()[0])
print(sorted(first.keys()))

## 6. Offline fallback — download JSONL

**Most reliable path.** Download → local machine → Vite:

```bash
# on your laptop
cp ~/Downloads/live.jsonl /path/to/SpecProbe/public/traces/live.jsonl
npm run dev
# open UI → "indent drift · ngram vs eagle" or drop the file
```

In [ ]:
from google.colab import files

files.download(JSONL_PATH)
print("Downloaded. Drop into local SpecProbe UI (file picker or public/traces/live.jsonl).")

## 7. Optional — FastAPI bridge + Colab native port forward

Uses Colab's built-in `output.serve_kernel_port_as_window(8787)` (no ngrok required).

Then point your **local** Vite app at the forwarded URL, or open `/bundle` in the Colab window to sanity-check JSON.

In [ ]:
import threading
import uvicorn

# Point the bridge at the JSONL we just wrote
os.environ["SPECPROBE_JSONL"] = JSONL_PATH

# Re-apply nest_asyncio before uvicorn (Colab already has a loop)
import nest_asyncio
nest_asyncio.apply()

from specprobe_hook.bridge import app


def run_bridge() -> None:
    uvicorn.run(app, host="0.0.0.0", port=BRIDGE_PORT, log_level="info")


t = threading.Thread(target=run_bridge, daemon=True)
t.start()

import time, urllib.request
for _ in range(40):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{BRIDGE_PORT}/health", timeout=1) as r:
            print(r.read().decode())
            break
    except Exception:
        time.sleep(0.25)
else:
    raise RuntimeError("bridge failed to start")

from google.colab import output
output.serve_kernel_port_as_window(BRIDGE_PORT)
print(f"Colab is exposing localhost:{BRIDGE_PORT} in a window.")
print("Hit /health, /events, /bundle in that window — or tunnel that URL into local Vite.")

## 8. Optional — n-gram vs EAGLE mini sweep (two JSONL files)

Produces two files you can load together locally for the side-by-side lane. Skips EAGLE if `EAGLE_MODEL is None`.

In [ ]:
sweep_dir = Path("/content/specprobe_traces/sweep")
sweep_dir.mkdir(parents=True, exist_ok=True)

runs = [("ngram", None)]
if EAGLE_MODEL:
    runs.append(("eagle", EAGLE_MODEL))

for strategy, eagle in runs:
    path = sweep_dir / f"{strategy}.jsonl"
    path.write_text("", encoding="utf-8")
    os.environ["SPECPROBE_JSONL"] = str(path)
    os.environ["SPECPROBE_STRATEGY"] = strategy
    os.environ["SPECPROBE_RUN_ID"] = f"colab_{strategy}"
    assert install(force=True)
    llm = build_llm(enable_spec=True, eagle_model=eagle)
    _ = greedy_generate(llm, PROMPT, max_tokens=MAX_TOKENS)
    nuke_llm(llm)
    time.sleep(1.0)
    print(strategy, "→", path, "lines", sum(1 for l in path.read_text().splitlines() if l.strip()))

# bundle into one downloadable jsonl for the UI compare
bundle = sweep_dir / "ngram_vs_eagle.jsonl"
with bundle.open("w", encoding="utf-8") as out:
    for strategy, _ in runs:
        out.write((sweep_dir / f"{strategy}.jsonl").read_text(encoding="utf-8"))
        if not (sweep_dir / f"{strategy}.jsonl").read_text(encoding="utf-8").endswith("\n"):
            out.write("\n")

from google.colab import files
files.download(str(bundle))
print("Downloaded ngram_vs_eagle.jsonl — drop into the SpecProbe file picker for side-by-side.")

---

### Checklist

- [ ] GPU runtime enabled
- [ ] `nest_asyncio.apply()` ran before uvicorn / heavy async work
- [ ] Losslessness assert passed
- [ ] `live.jsonl` has ≥1 event and downloaded
- [ ] Local Vite renders the lane / side-by-side

If losslessness fails, file a bug — the hook must never mutate sampler tensors.